# FINS5557 project notebook

Our one notebook for exploring the data and developing the model. Code that works and gets reused (loading, features, training, SHAP, fairness, PSI) is moved into `src/`, and this notebook imports it, so the app and the notebook always use the same steps.

| Section | Status |
|---|---|
| 0. Setup | done |
| 1. Data: exploration and checks | done |
| 2. Features | done |
| 3. Models: baseline vs random forest vs XGBoost | done |
| 4. Calibration and threshold | done |
| 5. Explainability (SHAP) and fairness | to do |

See `docs/BUILD_PLAN.md` for the full plan.

**Before you run:** activate `.venv`, pick it as this notebook's kernel (top right, **Select Kernel**), then run top to bottom. Clear outputs before committing.

## 0. Setup
### 0.1 Install packages
Installs `requirements-dev.txt` (which includes `requirements.txt`). Skip if you've already installed them.

In [ ]:
%pip install -q -r ../requirements-dev.txt

### 0.2 Imports and package versions
Prints each version, so a broken setup shows up here rather than half-way through.

`autoreload` picks up changes to `src/` files. If you still get an import error after code changes, use **Restart** in the notebook toolbar, then **Run All**.

In [ ]:
# Reload src/ modules automatically when their files change, so edits to src/
# show up without restarting the kernel.
%load_ext autoreload
%autoreload 2

import sys
import warnings
from importlib.metadata import version
from pathlib import Path

# Make `src` importable whether the notebook runs from the project root or from notebooks/
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.config import GOOGLE_API_KEY, LLM_BACKEND, PROTECTED, RANDOM_STATE, TARGET
from src.data import age_years, clean, features_target, load_raw, split

warnings.filterwarnings("ignore", category=FutureWarning)
np.random.seed(RANDOM_STATE)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.3f}".format)

print("Python", sys.version.split()[0])
for pkg in ["pandas", "numpy", "scikit-learn", "xgboost", "shap", "streamlit", "python-dotenv", "joblib"]:
    print(f"{pkg:15s} {version(pkg)}")

### 0.3 Settings check
Reports only whether an API key was found, never the key itself.

In [ ]:
print("Random seed:", RANDOM_STATE)
print("LLM backend:", LLM_BACKEND)
print("GOOGLE_API_KEY found:", bool(GOOGLE_API_KEY))

## 1. Data: exploration and checks

**Data:** Home Credit Default Risk (Kaggle, 2018), `application_train.csv`: 307,511 consumer loan applications, one row each, with what the lender knew at the time of application. `TARGET` (renamed `default`) = 1 if the client had payment difficulties on the loan. Source: https://www.kaggle.com/competitions/home-credit-default-risk/data (competition rules: not redistributed, so `data/raw/` is gitignored).

We load only the columns we use (inputs listed in `src/config.py`, plus the ID, target and the protected attributes for the fairness audit).

**Findings** (fill in after running; this becomes the Data section of the report):
- Default rate: **8.1%** (24,825 of 307,511 applications had payment difficulties). Classes are imbalanced about 11:1, so accuracy is meaningless (predicting "no default" for everyone scores 91.9%); we use AUC, Brier score and profit instead. The stratified 60/20/20 split keeps 8.1% in train (184,506 rows), validation (61,502) and test (61,503).
- Strongest early signals: the three **external credit scores**. Moving from the lowest to the highest fifth of `EXT_SOURCE_3` cuts the default rate from 16.4% to 3.4% (`EXT_SOURCE_2`: 15.2% to 3.6%; `EXT_SOURCE_1`: 14.6% to 3.0%). Among categories, lower secondary education (10.9%), renting or living with parents (11.7-12.3%) and "Working" income type (9.6%) are riskier; pensioners (5.4%) and revolving loans (5.5%) are safer.
- Missing data and what it means (thin files): `EXT_SOURCE_1` is missing for 56.4% of applicants and `EXT_SOURCE_3` for 19.8%; `OWN_CAR_AGE` is missing for 66% (no car), `OCCUPATION_TYPE` for 31%. A missing `EXT_SOURCE_3` comes with a higher default rate (9.3% vs 8.1% overall), consistent with a thin credit file, so gaps are kept as information (median fill plus a "was missing" flag).
- Data issues and how we handle them: `DAYS_EMPLOYED = 365243` for 18.0% of rows is Home Credit's "not employed" code (in practice these are all pensioners), turned into a flag. `CODE_GENDER` has 4 rows coded `XNA`, set to missing (audit only). Income has extreme outliers (max 117m; 0.08% above 1m), which the tree models are robust to. No duplicate application IDs.
- Fairness preview: default rates differ by protected group: men 10.1% vs women 7.0%; age falls steadily from 11.5% (20s) to 4.9% (60s); single / civil marriage 9.8-9.9% vs married 7.6%. Gender, age and family status are never model inputs, but other inputs (income type, occupation, education) can act as proxies, so the Step 10 audit checks approval rates by these groups. Education (an input) ranges from 1.8% (academic degree, only 164 people) to 10.9% (lower secondary).
- Who is in the data (selection bias): every row is a loan Home Credit **already approved**. Applicants it rejected have no outcome, so the data starts after one credit check has filtered out the weakest applicants. This is the classic **reject-inference** problem in credit scoring. It shows in patterns that reflect the lender's past decisions rather than pure risk: longer loans default *less* (section 2), probably because long terms were only granted to strong applicants. The likely effect is a model that is too lenient on the weakest applicants if used on everyone who applies. Proper reject inference would need the rejected applications with the same inputs; the refused applications in `previous_application.csv` lack most of them (including the external scores). So the model is scoped as a **second-stage screen** for applicants who already pass the lender's basic policy rules, inputs outside the training range are referred to an analyst (Step 7), and PSI monitoring flags a population unlike the training data (Step 11).

### 1.1 Load the data
Reads `data/raw/application_train.csv`. If the file is missing, the error says where to download it.

In [ ]:
raw = load_raw()
print(raw.shape)  # expect (307511, 27): ID, target, 23 inputs... see config for the exact list
raw.head()

### 1.2 Target: how many applicants default?
About 8% is expected. That is a realistic consumer-lending rate (the Taiwan data had 22%). Predicting "no default" for everyone would score about 92% accuracy, which is why we judge models on AUC, KS and Brier score and use class weights.

In [ ]:
raw[TARGET].value_counts(normalize=True).rename("share")

### 1.3 Data quality checks
Home Credit has many missing values. Missing doesn't always mean bad data: no external credit score means the applicant has little credit history (a thin file).

In [ ]:
missing = raw.isna().mean().sort_values(ascending=False)
print("Share missing per column (columns with any gaps):")
print(missing[missing > 0].round(3).to_string())
print()
print("Duplicate application IDs:", int(raw["SK_ID_CURR"].duplicated().sum()))
print("DAYS_EMPLOYED = 365243 ('not employed' code):", f"{(raw['DAYS_EMPLOYED'] == 365243).mean():.1%}")
print("CODE_GENDER values:", raw["CODE_GENDER"].value_counts().to_dict())

In [ ]:
df = clean(raw)
print("After cleaning, CODE_GENDER:", df["CODE_GENDER"].value_counts(dropna=False).to_dict())

### 1.4 External scores vs default
`EXT_SOURCE_1/2/3` are normalised scores from external data sources (credit bureaus). Expect them to be the strongest predictors, and expect applicants with **no** score to default more.

In [ ]:
for col in ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]:
    by_quintile = df.groupby(pd.qcut(df[col], 5), observed=True)[TARGET].mean()
    print(col, "| missing:", f"{df[col].isna().mean():.1%}",
          "| default rate if missing:", f"{df.loc[df[col].isna(), TARGET].mean():.1%}")
    print(by_quintile.round(3).to_string(), "\n")

### 1.5 Amounts and affordability
Income and loan amounts are very skewed (a few huge incomes), which matters for logistic regression (trees don't mind). The repayment-to-income ratio is the classic affordability measure.

In [ ]:
df[["AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "AMT_GOODS_PRICE"]].describe().T

In [ ]:
affordability = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
df.groupby(pd.qcut(affordability, 5), observed=True)[TARGET].mean().rename("default_rate by repayment/income")

In [ ]:
# Categorical inputs: default rate by category (largest groups first)
for col in ["NAME_CONTRACT_TYPE", "NAME_INCOME_TYPE", "NAME_EDUCATION_TYPE", "NAME_HOUSING_TYPE"]:
    print(df.groupby(col, dropna=False)[TARGET].agg(applicants="size", default_rate="mean")
            .sort_values("applicants", ascending=False).round(3), "\n")

### 1.6 Fairness preview
Gender, age and family status are **not** model inputs, but we audit the model's decisions by them later. Here are the raw default rates by group as a baseline.

In [ ]:
age_band = pd.cut(age_years(df), [20, 30, 40, 50, 60, 70], right=False)
for name, group in [("Gender", df["CODE_GENDER"]), ("Age band", age_band),
                    ("Family status", df["NAME_FAMILY_STATUS"]), ("Education", df["NAME_EDUCATION_TYPE"])]:
    print(name)
    print(df.groupby(group, observed=True, dropna=False)[TARGET].agg(applicants="size", default_rate="mean").round(3), "\n")

### 1.7 Train / validation / test split
Stratified 60/20/20, done before any model fitting. The default rate should be the same in all three parts.

In [ ]:
train, val, test = split(df)
pd.DataFrame({
    "rows": [len(train), len(val), len(test)],
    "default_rate": [train[TARGET].mean(), val[TARGET].mean(), test[TARGET].mean()],
}, index=["train", "validation", "test"])

In [ ]:
X_train, y_train = features_target(train)
print(X_train.shape[1], "model inputs; protected attributes removed:", PROTECTED)
list(X_train.columns)

## 2. Features

New features built by `add_features()` in `src/features.py`, each a standard credit-risk idea:

| Feature | Definition | Credit-risk meaning |
|---|---|---|
| `credit_income_ratio` | loan / annual income | Leverage: how big the loan is for this income |
| `annuity_income_ratio` | repayment / income | Affordability (debt-service ratio), what responsible-lending checks look at |
| `credit_term` | loan / repayment | Roughly the number of repayments; longer loans are riskier |
| `goods_credit_ratio` | goods price / loan | Below 1 = borrowing extra cash on top of the goods |
| `years_employed`, `not_employed` | from `DAYS_EMPLOYED`; 365243 = not employed | Job stability; the code must not be read as 1,000 years in a job |
| `ext_source_mean` | mean of the available external scores | Overall external credit view |
| `ext_source_missing` | number of missing external scores (0-3) | Thin-file signal |
| `income_per_person` | income / family members | Income spread over the household |

Divisions are safe: a zero, negative or missing denominator gives a missing value, which the pipeline fills with the training median and flags with a "was missing" column. Everything here uses the **training split only**.

**Findings** (fill in after running):
- Strongest engineered feature: **`ext_source_mean`** (correlation with default -0.225; default rate 18.8% in the lowest fifth vs 2.5% in the highest), far ahead of the rest. Next are `years_employed` (-0.077; 11.3% for under 1.7 years in the job vs 5.2% for over 10 years) and `goods_credit_ratio` (-0.064; 11.7% when the loan is well above the goods price, i.e. extra cash borrowed, vs 4.2% when the goods cost more than the loan).
- Shape of the affordability relationship: **weak and flat at the top**. Default rises only from 7.2% (repayment under 10% of income) to 8.7-8.8% around 19-25%, then levels off (8.4% above 25%). `credit_income_ratio` is hump-shaped (7.3% → 8.8% → 7.2%). Likely reason: the lender already limited loan size by income when approving, so stated affordability adds little on its own. `credit_term` is also non-monotonic: loans of 14-22 repayments default most (9.7-9.9%) and the longest loans least (6.1-6.6%), against the textbook expectation, probably because longer terms were granted to stronger applicants. The tree models can use these non-linear shapes; logistic regression cannot.
- Thin-file effect (`ext_source_missing`): default rises from 7.4% with all three external scores to 8.1% with one missing and 10.0% with two missing (the 109 applicants with none are too few to read). Separately, `not_employed` applicants default *less* (5.5% vs 8.6%) because they are pensioners, so this flag mostly captures age; worth noting for the fairness audit.

In [ ]:
from src.config import ENGINEERED_FEATURES
from src.features import add_features, build_preprocessor

train_f = add_features(train)
train_f[ENGINEERED_FEATURES].describe().T

### 2.1 Each feature against the default rate
Correlation gives a quick ranking; the quintile tables show the shape of each relationship (a correlation near 0 can still hide a U-shape).

In [ ]:
train_f[[*ENGINEERED_FEATURES, TARGET]].corr()[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).rename("correlation")

In [ ]:
for col in ["annuity_income_ratio", "credit_income_ratio", "credit_term", "goods_credit_ratio", "years_employed", "ext_source_mean"]:
    print(train_f.groupby(pd.qcut(train_f[col], 5, duplicates="drop"), observed=True)[TARGET].mean().round(3).rename(col), "\n")
print(train_f.groupby("ext_source_missing")[TARGET].agg(applicants="size", default_rate="mean").round(3))
print(train_f.groupby("not_employed")[TARGET].agg(applicants="size", default_rate="mean").round(3))

### 2.2 Preprocessing pipeline
`build_preprocessor()`: numeric columns get median imputation plus a 0/1 "was missing" column, then scaling (needed for logistic regression, harmless for trees). Categorical columns get "Unknown" for missing, rare categories (< 1% of training rows) grouped, then one-hot encoding. Fitted on the training split only, then applied unchanged to validation. Protected attributes and the ID are dropped. Pattern adapted from Code Bank W06-04.

In [ ]:
pre = build_preprocessor()
X_train_ready = pre.fit_transform(train_f)
X_val_ready = pre.transform(add_features(val))
print("train:", X_train_ready.shape, " validation:", X_val_ready.shape)
list(pre.get_feature_names_out())

## 3. Models: baseline vs random forest vs XGBoost

Training lives in `src/train.py`. Run it once from the project root (about 25 minutes on the full data with the model v2 search):

```bash
python -m src.train
```

It tunes each model with 3-fold stratified cross-validation on the **training split only** (model v2, Step 5c: a randomised search of 30 settings per tree model and a full grid of 10 for logistic regression, scored on AUC and PR-AUC; every candidate is saved in `reports/tuning_results.csv`), scores all three on the **validation split**, applies the model-choice rule and saves `models/model.joblib`, `models/metadata.json` and `reports/model_comparison.csv`. This section reads those files instead of retraining. The test split is still untouched.

| Model | Why it's in the comparison | Imbalance handling |
|---|---|---|
| Logistic regression | Traditional scorecard baseline (Code Bank W01-06, W05-02): linear, transparent, easy to defend to a regulator | Tuned: `class_weight` none or `"balanced"` |
| Random forest | Many decision trees averaged: captures non-linear patterns and interactions | Tuned: `class_weight` none, `"balanced"` or `"balanced_subsample"` |
| XGBoost | Boosted trees, usually the strongest on tabular credit data (Code Bank W02-06, W05-03) | Tuned: `scale_pos_weight` 1 (none), √11.4 ≈ 3.4 (partial) or 11.4 (full: non-defaulters / defaulters) |

Every model is one pipeline (feature engineering → scaling / one-hot → model), so features are fitted inside each CV fold and the saved model takes raw applicant columns.

**Model-choice rule** (`MIN_AUC_GAIN` in `src/config.py`): take the highest validation AUC, but a tree model has to beat logistic regression by at least 0.01 AUC to justify the loss of transparency.

**Findings** (fill in after retraining on Home Credit):
- Model comparison (validation split, 61,502 applicants, model v2): XGBoost AUC **0.757** (Gini 0.513, KS 0.380, PR-AUC 0.240), random forest 0.745 (PR-AUC 0.227), logistic regression 0.740 (PR-AUC 0.219). The PR-AUCs look low but should be read against the 8% default rate (a random model scores 0.08), so XGBoost puts defaulters near the top about three times better than chance. At each model's balanced-accuracy cut-off XGBoost catches 66% of defaulters while approving 72% of good customers. Brier scores here (0.09-0.20) are for the uncalibrated scores; section 4 calibrates.
- Chosen model and why: **XGBoost** (486 trees, `learning_rate` 0.036, `max_depth` 5, `min_child_weight` 20, `subsample` 0.78, `colsample_bytree` 0.78, `scale_pos_weight` 3.4). Its validation AUC is 0.017 above logistic regression, which clears the 0.01 margin we set for giving up a fully transparent model.
- Overfitting check (CV vs validation AUC): logistic regression 0.746 vs 0.740, random forest 0.750 vs 0.745, XGBoost 0.762 vs 0.757, test 0.757. Gaps of at most 0.006: the models generalise to unseen applicants.
- v1 vs v2 (3.2): the proper search **barely changed the result**. XGBoost validation AUC 0.757 in both, test AUC 0.756 → 0.757, test PR-AUC 0.231 → 0.236, test Brier 0.068 in both; the random forest gained 0.002. Realised test profit is 0.1% lower (3.625bn → 3.622bn) because the cut-off moved from 22.5% to 21.0% (section 4). So v1's small grid was already close to the best these models can do: the limit is the information in application data (bureau and previous-application tables would be the way to improve), not the tuning. Worth reporting because we checked instead of assuming.
- What the tuning found (3.3): XGBoost candidates range from 0.739 to 0.762 CV AUC; the weak ones use a high learning rate (0.15-0.19) with hundreds of trees, which overfits (CV AUC falls as the learning rate rises, correlation -0.58), while the best use about 0.04; the top three settings are within 0.001 of each other. The random forest did best with `max_features="sqrt"` (the default; trying more inputs per split, the exam's larger `mtry`, didn't help), leaves of at least 20 and deep trees (depth 16 or unlimited). Logistic regression is flat (0.745-0.746) across `C`.
- Did class weighting matter? **No, not once the model is calibrated.** Best CV AUC by imbalance setting: XGBoost no weighting 0.761, partial 0.762, full 0.760; random forest none 0.750 vs balanced 0.750; logistic regression 0.746 either way. All within one standard deviation across folds (about 0.003). Weighting only shifts the raw scores up (uncalibrated Brier 0.193 with v1's full weighting, 0.088 with v2's partial), and calibration removes that shift (0.068 in both). Here calibration plus a profit cut-off do the imbalance work, so an unweighted model would be equally fine; the search kept partial weighting only because it won by 0.001.

In [ ]:
import json

import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve

from src.config import METADATA_PATH, REPORTS_DIR

comparison = pd.read_csv(REPORTS_DIR / "model_comparison.csv", index_col="model")
metadata = json.loads(METADATA_PATH.read_text())
print("Chosen:", metadata["chosen_model"], "|", metadata["reason"])
print("Best settings:", metadata["best_params"])
comparison

### 3.1 ROC curves on the validation split
Each curve shows the trade-off between catching defaulters (true positive rate) and wrongly flagging good customers (false positive rate) across all cut-offs. A higher curve means better ranking; the dashed line is a coin flip (AUC 0.5).

In [ ]:
val_preds = pd.read_csv(REPORTS_DIR / "validation_predictions.csv")

fig, ax = plt.subplots(figsize=(6, 5))
for name in comparison.index:
    fpr, tpr, _ = roc_curve(val_preds[TARGET], val_preds[name])
    ax.plot(fpr, tpr, label=f"{name} (AUC {comparison.loc[name, 'auc']:.3f})")
ax.plot([0, 1], [0, 1], "--", color="grey", label="random (AUC 0.500)")
ax.set(xlabel="False positive rate (good customers flagged)",
       ylabel="True positive rate (defaulters caught)", title="ROC curves: validation split")
ax.legend(loc="lower right")
plt.show()

### 3.2 Model v1 vs v2 (Step 5c)
v1 (tag `v1-home-credit-baseline`, artefacts in `archive/v1-home-credit-baseline/`) used small fixed grids and fixed class weights. v2 uses a randomised search that also tunes the **imbalance correction** (none, partial or full class weighting), as the Big Data exam tuned its forest's `sampsize`, plus the forest's `max_features` (the exam's `mtry`). Same data, same split and the same validation and test rows, so the numbers compare directly. The v1 model is re-scored on the test split only to add metrics it didn't report (PR-AUC); nothing is chosen on the test split.

In [ ]:
import joblib
from sklearn.metrics import average_precision_score

from src.config import ROOT

v1_dir = ROOT / "archive" / "v1-home-credit-baseline"
v1 = json.loads((v1_dir / "metadata.json").read_text())
X_test, y_test = features_target(test)
v1_test_pd = joblib.load(v1_dir / "model.joblib").predict_proba(X_test[v1["input_columns"]])[:, 1]


def summary(meta, test_pr_auc):
    t = meta["test_metrics"]
    return {
        "chosen model": meta["chosen_model"],
        "settings": meta["best_params"],
        "CV AUC": meta["validation_metrics"]["cv_auc"],
        "validation AUC": meta["validation_metrics"]["auc"],
        "test AUC": t["auc"],
        "test PR-AUC": test_pr_auc,
        "test Brier": t["brier"],
        "calibration": meta["calibration"]["method"],
        "cut-off": meta["threshold"],
        "approval rate": t["approval_rate"],
        "default rate among approved": t["default_rate_approved"],
        "realised profit (test)": t["realised_profit"],
    }


versions = pd.DataFrame({
    "v1": summary(v1, average_precision_score(y_test, v1_test_pd)),
    "v2": summary(metadata, metadata["test_metrics"]["pr_auc"]),
})
print("Validation AUC per model:")
print(pd.DataFrame({"v1": {m: r["auc"] for m, r in v1["all_models"].items()},
                    "v2": {m: r["auc"] for m, r in metadata["all_models"].items()}}).round(4))
versions

### 3.3 What the tuning found
Every candidate's cross-validated scores are in `reports/tuning_results.csv` (30 random settings per tree model, all 10 for logistic regression). Two questions: which settings mattered, and did class weighting still help? Class weighting changes how the model is trained, but after calibration only the **ranking** matters, so the fair test is CV AUC with and without it.

In [ ]:
tuning = pd.read_csv(REPORTS_DIR / "tuning_results.csv")
print("Spread of CV AUC across the candidates tried:")
print(tuning.groupby("model")["cv_auc"].agg(["count", "min", "median", "max"]).round(4), "\n")

imbalance = {"logistic_regression": "class_weight", "random_forest": "class_weight", "xgboost": "scale_pos_weight"}
for model, col in imbalance.items():
    t = tuning[tuning["model"] == model]
    setting = t[col].map(lambda v: "none" if pd.isna(v) or v in (1, 1.0) else (f"{v:.1f}" if isinstance(v, float) else v))
    print(f"{model}: best CV AUC by imbalance setting ({col})")
    print(t.groupby(setting)["cv_auc"].agg(["count", "max", "median"]).round(4), "\n")

tuning.sort_values("cv_auc", ascending=False).groupby("model").head(3)

## 4. Calibration and threshold

Code in `src/threshold.py`, run as part of `python -m src.train`. Three steps turn a model score into a credit decision:

1. **Calibrate.** Class weighting makes the chosen model rank customers well but overstate risk (its average score is well above the ~8% default rate). A calibrator fitted on the validation split maps scores to **PDs** that match observed default rates. Sigmoid (Platt) and isotonic calibration are compared on out-of-fold Brier score, so neither is judged on rows it was fitted on.
2. **Choose the cut-off from the lender's economics**, not 0.5. Approving a customer earns `MARGIN × EAD` if they repay and loses `LGD × EAD` if they default, so the expected loss is `PD × LGD × EAD` (the Basel formula). Assumptions in `src/config.py`: LGD = 60%, net margin over the loan's life = 15%, EAD = loan amount. The break-even PD is `margin / (margin + LGD)` = 20%. The cut-off is the PD that maximises **realised** profit on the validation split.
3. **Decide:** approve below `cut-off - 2 points`, decline above `cut-off + 2 points`, refer the band in between to an analyst (`REFER_BAND` in `src/config.py`).

The test split is then scored **once**, at the chosen cut-off.

**Findings** (fill in after retraining on Home Credit):
- Calibration: the partially weighted XGBoost scores still overstate risk (validation Brier 0.088 uncalibrated; v1's fully weighted scores were at 0.193). Isotonic and sigmoid calibration fix this equally (Brier 0.0681 both; isotonic kept by a hair). The reliability curve lies on the diagonal, and the mean calibrated PD on the test split is **8.2%, against an actual default rate of 8.1%**, so the PDs can be used directly in expected-loss calculations.
- Cut-off vs break-even PD (20%): the profit-maximising cut-off on validation is **21.0%** (v1: 22.5%), very close to the 20% break-even PD from `margin / (margin + LGD)`. The profit curve is flat between 20% and 25% (3.59bn vs 3.58bn), so theory and realised profit agree when the PDs are well calibrated.
- Test results (61,503 applicants, scored once at 21.0%): AUC **0.757** (validation 0.757), Gini 0.515, KS 0.384, PR-AUC 0.235, Brier 0.068. The model approves 93.1%; the default rate among approved loans falls to **6.5% from 8.1%**. It declines 1,264 of 4,965 defaulters (TPR 25.5%) and wrongly declines 2,953 of 56,538 good customers (TNR 94.8%); 30% of those declined would really have defaulted (precision), almost four times the 8% base rate.
- Profit vs balanced-accuracy cut-off (4.5): the exam's balanced-accuracy rule picks **8.7%** on the same validation PDs. On the test split it catches far more defaulters (TPR 67.1% vs 25.5%) and scores a much better balanced accuracy (0.692 vs 0.601), but approves only 68.1% of applicants instead of 93.1% and earns **12% less profit** (3.18bn vs 3.62bn). The reason is the implied price of a missed defaulter: balanced accuracy treats it as 11.4 wrongly declined good customers, the lender's economics as 4 (LGD 60% / margin 15%). Turning away a quarter of the book to avoid defaults costs more margin than it saves in losses. The app keeps the profit cut-off; balanced accuracy is the right rule only when both errors are equally costly per *share* of each class, which a lender's errors are not.
- Refer band share: **2.2%** of validation applicants at ±2 points (approve 92.3%, decline 5.5%; v1: 3.2%). The 28 Oct decision to keep ±2 still holds: it keeps analyst workload small and focused on genuinely borderline cases.

In [ ]:
from sklearn.calibration import calibration_curve

from src.threshold import break_even_pd, decide

print("Calibration method:", metadata["calibration"]["method"])
print("Validation Brier:", metadata["calibration"]["validation_brier"])
print("Assumptions:", metadata["assumptions"])
print(f"Cut-off: decline at PD >= {metadata['threshold']:.3f}, refer band +/- {metadata['refer_band']}")

### 4.1 Reliability curve
Customers grouped into 10 bins by predicted risk: average prediction (x) against the share that actually defaulted (y). A calibrated model sits on the diagonal. Validation split, out-of-fold calibrated PDs.

In [ ]:
chosen = metadata["chosen_model"]
fig, ax = plt.subplots(figsize=(6, 5))
for label, scores in [(f"{chosen}, uncalibrated", val_preds[chosen]), ("calibrated PD", val_preds["calibrated_pd"])]:
    observed, predicted = calibration_curve(val_preds[TARGET], scores, n_bins=10, strategy="quantile")
    ax.plot(predicted, observed, "o-", label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.set(xlabel="Predicted probability of default", ylabel="Observed default rate",
       title="Reliability curve: validation split")
ax.legend()
plt.show()

### 4.2 Profit curve
Profit on the validation split if everyone below each cut-off is approved. **Realised** uses what actually happened; **expected** uses the PDs. Too low a cut-off turns away profitable customers, too high a cut-off approves loss-making ones.

In [ ]:
curve = pd.read_csv(REPORTS_DIR / "profit_curve.csv")

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(curve["threshold"], curve["realised_profit"] / 1e6, label="realised profit")
ax.plot(curve["threshold"], curve["expected_profit"] / 1e6, label="expected profit (from PDs)")
ax.axvline(metadata["threshold"], color="black", label=f"chosen cut-off ({metadata['threshold']:.3f})")
ax.axvline(break_even_pd(), color="grey", linestyle="--", label=f"break-even PD ({break_even_pd():.3f})")
ax.axvspan(metadata["threshold"] - metadata["refer_band"], metadata["threshold"] + metadata["refer_band"],
           color="orange", alpha=0.15, label="refer band")
ax.set(xlabel="PD cut-off (approve below)", ylabel="Profit, million (loan currency)", title="Profit by cut-off: validation split")
ax.legend()
plt.show()

curve.set_index("threshold").loc[[0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]]

### 4.3 How wide should the refer band be?
Share of validation applicants sent to an analyst for different band widths around the cut-off.

In [ ]:
pds = val_preds["calibrated_pd"]
pd.DataFrame({
    band: pd.Series(decide(pds, metadata["threshold"], band)).value_counts(normalize=True)
    for band in [0.01, 0.02, 0.03, 0.05]
}).T.rename_axis("refer band (+/-)").round(3)[["approve", "refer", "decline"]]

### 4.4 Final test-set results (reported once)
The confusion matrix treats "decline" as predicting default (refer band ignored).

In [ ]:
test_metrics = json.loads((REPORTS_DIR / "test_metrics.json").read_text())
cm = test_metrics["confusion_matrix"]
print(pd.DataFrame([[cm["approved_repaid"], cm["declined_repaid"]], [cm["approved_defaulted"], cm["declined_defaulted"]]],
                   index=["actually repaid", "actually defaulted"], columns=["approved", "declined"]))
print()
pd.Series({k: v for k, v in test_metrics.items() if not isinstance(v, dict)}).round(4)

### 4.5 Two ways to choose the cut-off: profit vs balanced accuracy
The Big Data exam chose its cut-off by **balanced accuracy**, (TPR + TNR) / 2 (Lecture 2, slide 57). We choose it by **profit**. Both are found on the same out-of-fold calibrated validation PDs (`src/threshold.py`, `compare_thresholds`) and reported here on the test split. The app keeps the profit cut-off.

Why they differ: balanced accuracy gives the two classes equal weight, so with 8% defaulters one missed defaulter counts as much as (1 - 0.08) / 0.08 ≈ **11** wrongly declined good customers. The lender's economics say a missed defaulter loses LGD = 60% of the loan, while a wrongly declined good customer forgoes the 15% margin: a ratio of **4**. Balanced accuracy therefore overweights defaults for this lender and declines more people than is profitable.

In [ ]:
thresholds = pd.read_csv(REPORTS_DIR / "threshold_comparison.csv", index_col="rule")
default_rate = y_test.mean()
print(f"Implied cost of a missed defaulter vs a wrongly declined good customer:")
print(f"  balanced accuracy: (1 - {default_rate:.3f}) / {default_rate:.3f} = {(1 - default_rate) / default_rate:.1f}")
print(f"  lender's economics: LGD / margin = {metadata['assumptions']['lgd']} / {metadata['assumptions']['margin']} "
      f"= {metadata['assumptions']['lgd'] / metadata['assumptions']['margin']:.1f}")
thresholds

### 4.6 Scoring applicants (Step 5: `src/predict.py`)
This is the interface the app uses. `load_artifacts()` loads the calibrated model and its metadata once (the app caches it with `st.cache_resource`). `predict()` scores one applicant from a dict of raw fields; `predict_batch()` scores a whole table in one vectorised call. Both use the cut-off and refer band saved in `models/metadata.json`, so the app can't drift from what was trained. Protected attributes are not needed: the applicant below is scored without gender, age or family status.

`reasons` and `shap` are empty until the explainability step (section 5) is built.

In [ ]:
from src.predict import load_artifacts, predict, predict_batch

artifacts = load_artifacts()

applicant = features_target(test)[0].iloc[0].to_dict()  # only the fields the app collects
print("Applicant:", applicant)
predict(applicant, artifacts)

In [ ]:
# A missing field gives a clear error instead of a crash deep inside the model
incomplete = {k: v for k, v in applicant.items() if k != "AMT_CREDIT"}
try:
    predict(incomplete, artifacts)
except ValueError as err:
    print("ValueError:", err)

In [ ]:
# Score 1,000 test applicants at once and check the decision mix
scored = predict_batch(test.drop(columns=TARGET).head(1000), artifacts)
print(scored["decision"].value_counts(normalize=True).round(3))
scored[["AMT_CREDIT", "AMT_INCOME_TOTAL", "EXT_SOURCE_2", "pd", "decision"]].head()

## 5. Explainability (SHAP)

Code in `src/explain.py` (Code Bank W05-06 and W02-07). A lender that declines an applicant has to be able to say **why**: responsible-lending rules and model-risk guidance (e.g. SR 11-7, GDPR Art. 22) expect the main reasons behind an automated decision, and an XGBoost model has no coefficients to read them from. SHAP fills that gap: it splits one applicant's score into a contribution per input, so that base value + contributions = that applicant's score.

- Contributions are in **log-odds** of the uncalibrated model. Calibration only rescales the score without reversing any order, so an input that raises the score also raises the PD.
- Model columns are added back up to the 32 inputs an analyst knows (one-hot columns and "was missing" flags are merged into their input).
- `top_reasons()` turns them into plain-English factors (`REASON_PHRASES`) and only lists factors that **raised** the risk. Phrases name the factor, not a direction ("Time in current job"), because several effects are U-shaped or reversed and a decline reason must never be false.

**Findings** (fill in after running):
- Global drivers (model v2): the **external credit scores** dominate (average |SHAP| 0.47 for `ext_source_mean`, plus 0.16 for `EXT_SOURCE_3`, 0.08 for `EXT_SOURCE_1` and 0.07 for `EXT_SOURCE_2`). Next comes **loan structure**: `credit_term` 0.19, `goods_credit_ratio` 0.12, repayment size 0.09 and goods price 0.08, then time in job (0.07). Education is 5th (0.10), but mostly *lowers* the score for graduates rather than raising it for others (see below).
- Most common decline reasons (all 4,149 validation applicants at or above the 21.0% cut-off): external credit scores **97%**, length of the loan 55%, amount borrowed beyond the price of the goods 37%, repayments relative to income 24%, time in current job 21%, size of the loan 14%. Education level appears in 3.2% (0.9% with v1) and protected attributes never can; the fairness audit (Step 10) should still check education. Declines are driven by credit history and loan structure, so a lender could offer a counter-offer (a smaller loan, closer to the goods price) instead of a flat decline.
- Worked applicant (5.3): PD 36.7%, declined. The score starts at the average (-1.22 log-odds, about 23% before calibration; v1 started near 0, about 50%, because of its full class weighting) and ends at 0.55. The largest single push is borrowing well beyond the goods price (+0.33), but the external-score inputs together add more (+0.28 for `EXT_SOURCE_3`, +0.25 for the average), so the reasons read: external credit scores, amount borrowed beyond the price of the goods, repayments relative to income.

### 5.1 Which inputs drive the PD overall?
Average size of each input's contribution over 2,000 validation applicants. This shows importance, not direction.

In [ ]:
from src.explain import REASON_PHRASES, explain, top_reasons

explainer = artifacts["explainer"]
X_val, _ = features_target(val)
sample = X_val[metadata["input_columns"]].sample(2000, random_state=RANDOM_STATE)
contrib = explain(sample, explainer)

importance = contrib.abs().mean().sort_values()
ax = importance.tail(15).plot.barh(figsize=(7, 5), title="Mean |SHAP| per input: 2,000 validation applicants")
ax.set_xlabel("Average impact on the score (log-odds)")
plt.show()
importance.sort_values(ascending=False).head(10)

### 5.2 Which reasons would declined applicants get?
Top-3 reasons for every applicant in the sample at or above the cut-off: what decline letters would say most often.

In [ ]:
pds = artifacts["model"].predict_proba(sample)[:, 1]
declined = contrib[pds >= metadata["threshold"]]
reasons = pd.Series([r for _, row in declined.iterrows() for r in top_reasons(row)])
print(f"{len(declined)} of {len(sample)} sampled applicants at or above the {metadata['threshold']:.1%} cut-off")
(reasons.value_counts() / len(declined)).rename("share of declines citing it").head(10)

### 5.3 One applicant, explained
Waterfall for one declined applicant: starts at the average score and adds each input's contribution (Code Bank W02-07). Input values are left off so no real applicant data is shown.

In [ ]:
import shap

i = int(np.flatnonzero(pds >= metadata["threshold"])[0])
row = contrib.iloc[i]
shap.plots.waterfall(
    shap.Explanation(values=row.values, base_values=explainer["tree"].expected_value, feature_names=list(row.index)),
    max_display=10,
)
out = predict(sample.iloc[i].to_dict(), artifacts)
print(f"PD {out['pd']:.1%} -> {out['decision']}; reasons: {out['reasons']}")

## 6. Fairness

Code in `src/fairness.py` (Code Bank W04-06). Sex, age and family status are **never model inputs** (`PROTECTED` in `src/config.py`), but a model can still treat groups differently through inputs that are correlated with them (proxies). So the audit looks at the **decisions**, group by group, on the test split, using the automatic rule (approve when PD < cut-off). Education is a model input and is audited too.

| Measure | Question | Flagged when |
|---|---|---|
| Disparate-impact ratio | approval rate ÷ the most-approved group's | below 0.8 (four-fifths rule) |
| Demographic parity gap | how many points fewer are approved? | above 0.05 (CFPB safe harbour, W04-06) |
| Equalised-odds gap | are good customers, and defaulters, approved equally often? | above 0.05 |
| Mean PD vs default rate | are the PDs accurate for this group? | (reported, not flagged) |

Groups with fewer than 100 test applicants are listed but not judged. In the app's copy (`reports/fairness.csv`, written by `src/reporting.py`) their rates are removed, because a rate over one or a few people is their own data.

**Findings** (fill in after running):
- Disparate impact and parity: **no group falls below the four-fifths rule.** The lowest ratio is under-30s at 0.85 (approval 83.8% vs 98.6% for over-60s), then lower-secondary education 0.92, 30-39 and single applicants 0.93. Approval-rate gaps above 5 points: under 30 (14.7), lower secondary (7.4), 30-39 (7.1), single (6.8), secondary (6.0), incomplete higher (5.4), civil marriage (5.3). Men are approved 90.3% vs women 94.6% (4.3 points, within the 5-point threshold).
- Are the PDs fair (calibration within groups)? **Yes, closely.** In every judged group the mean PD is within 1.2 points of the actual default rate, so the score is not biased against any group: groups with lower approval rates really do default more (under 30: 11.6%; over 60: 4.8%). The largest gap is for under-30s, whose PDs are 1.1 points too high (12.7% vs 11.6%): small, but in the unfavourable direction for the group approved least.
- Equal opportunity (good customers approved): creditworthy applicants are **not** treated equally. 86.9% of under-30s who repaid were approved vs 98.8% of over-60s (11.9 points); men 92.5% vs women 95.9%; lower secondary 93.3% vs higher education 98.3%. The equalised-odds gaps are larger still because the model catches young defaulters far more often (approves 59.9% of under-30 defaulters vs 93.5% of over-60 defaulters). This is the known trade-off: when groups' default rates differ, a model can't be accurate within every group *and* have equal error rates; ours is accurate within groups.
- Proxies, how age gets in: the inputs most correlated with age are the "not employed" (pensioner) code (0.62) and `EXT_SOURCE_1` (0.61), then job tenure, children and registration age (about 0.3). Of the 0.99 log-odds score gap between under-30s and over-60s, the external scores (`ext_source_mean` and `EXT_SOURCE_3`) contribute 0.54, loan length 0.18, job tenure 0.12, time since the ID was renewed 0.06 and income type 0.05. So most of the age effect comes through the **external credit scores**, which reward a longer credit history.
- What to do about it (policy options for the report, not built): (1) accept and document it: the decisions follow accurately measured risk, and the inputs that carry age (credit scores, job tenure) have a clear credit rationale that can be explained and defended; (2) test removing the weakest proxies (`DAYS_ID_PUBLISH`, `DAYS_REGISTRATION`) and measure the cost in AUC and profit, a cheap experiment; (3) not recommended: different cut-offs by age or sex would equalise approval rates but use a protected attribute in the decision itself, which is direct discrimination. A fourth option is to send thin-file applicants to an analyst rather than decline them automatically, which helps young applicants without using age.
- **Decided (29 Oct): accept and document.** The decisions follow risk that is measured accurately within every group, and the inputs that carry age (external credit scores, job tenure) have a clear credit rationale. The model is not changed; the report explains the trade-off, the age proxies and the options considered.

### 6.1 The audit

In [ ]:
from src.fairness import fairness_table

X_test, y_test = features_target(test)
test_pd = artifacts["model"].predict_proba(X_test[metadata["input_columns"]])[:, 1]
audit = fairness_table(test, test_pd, metadata["threshold"])
audit[["attribute", "group", "n", "approval_rate", "di_ratio", "dp_gap", "good_customers_approved",
       "defaulters_approved", "eo_gap", "mean_pd", "default_rate", "flags"]].round(3)

### 6.2 Proxies: how does age get into the decision?
Age is not an input, yet approval rates fall steadily for younger applicants. Two checks: which inputs are correlated with age, and which inputs make the scores of under-30s and over-60s differ (average SHAP contribution per input, under-30s minus over-60s, on a sample of test applicants).

In [ ]:
from src.config import DAYS_EMPLOYED_SENTINEL, NUMERIC_INPUTS

age = age_years(test)
numeric = test[NUMERIC_INPUTS].assign(
    DAYS_EMPLOYED=test["DAYS_EMPLOYED"].where(test["DAYS_EMPLOYED"] != DAYS_EMPLOYED_SENTINEL),
    not_employed=(test["DAYS_EMPLOYED"] == DAYS_EMPLOYED_SENTINEL).astype(int),
)
print("Inputs most correlated with age (absolute correlation):")
print(numeric.corrwith(age).abs().sort_values(ascending=False).head(6).round(2), "\n")

sample = X_test[metadata["input_columns"]].sample(6000, random_state=RANDOM_STATE)
sample_age = age.loc[sample.index]
shap_by_input = explain(sample, artifacts["explainer"])
gap = (shap_by_input[sample_age < 30].mean() - shap_by_input[sample_age >= 60].mean()).sort_values(ascending=False)
print(f"Score gap, under 30 vs 60 and over: {gap.sum():.2f} log-odds. Largest contributions:")
gap.head(8).round(3)

## 7. Monitoring: population stability (PSI)

Code in `src/monitoring.py` (Code Bank W02-10, W05-07). A model is validated on the population it was trained on. Our training population is narrow (loans Home Credit had already approved, the selection bias from section 1), so if the applicants change, for example a campaign aimed at younger customers, the PDs can be wrong although the model hasn't changed. Supervisors expect lenders to monitor this (model risk management, e.g. SR 11-7).

For each input, PSI compares the share of applicants in each training bin (deciles, plus a bin for missing values and one for the "not employed" code) between a new batch and the training data: Σ (actual − expected) × ln(actual / expected). Below 0.10 stable, 0.10-0.25 watch, above 0.25 alert. The bins and training shares are stored in `models/reference_stats.json`, so the app's Monitoring page needs no training data. Categories rarer than 1% of training are pooled, as the model's encoder pools them.

The demo batches in `data/sample/` are **synthetic**: drawn from the training summary statistics, never copied from Kaggle rows.

**Findings** (fill in after running):
- Validation and test splits vs training: **all 23 inputs stable**, largest PSI 0.0005 (validation) and 0.0004 (test), as expected for a random split. A 1,000-applicant sample of the test split, the size of a realistic monthly batch, reaches only 0.023. So ordinary sampling noise stays far below the 0.10 watch line, and an alert on a batch this size is a real shift, not chance.
- Synthetic batches: the batch drawn like training is stable on every input (largest PSI 0.025). The shifted batch raises **8 alerts**: time since ID renewal (6.1), external score 2 (6.0, missing for 25% of applicants instead of 0.2%), employment (3.2), external score 3 (3.0), time since registration (2.6), external score 1 (1.1), housing (0.50) and goods price (0.27). Loan amount only reaches "watch" (0.15) although loans are 20% larger, because loan sizes are so widely spread that a 20% shift moves few applicants across a decile.
- Effect on the model's decisions: on the shifted batch the mean PD rises to **11.3%** (7.6% for the normal synthetic batch, 8.2% on the test split) and declines rise to 9.8% (3.9%). The model reacts to the riskier inputs as it should, but PSI can't say whether those PDs are still *accurate* for the new population: that needs the batch's repayment outcomes, months later. PSI is an early warning to review the model, not proof that it is wrong.
- Limits of PSI: (1) it checks one input at a time, so it can't see a change in how inputs combine; the normal synthetic batch shows this, since no input drifted yet approvals are 3 points higher than on the test split, because its inputs were drawn independently and lost their correlations; (2) it says nothing about accuracy, which needs outcomes (tracking the default rate by PD band as loans mature would be the next monitoring step); (3) the 0.10 / 0.25 thresholds are industry conventions, not statistical tests; (4) small batches are noisy, which is why categories rarer than 1% are pooled.

### 7.1 Real data: validation and test splits against training
These come from the same random split as training, so they should be stable. Also a 1,000-applicant test sample, the size of a realistic monthly batch.

In [ ]:
from src.monitoring import SAMPLE_PATHS, psi_table

reference = artifacts["reference"]
X_val_all, _ = features_target(val)
X_test_all, _ = features_target(test)
for name, batch in [("validation", X_val_all), ("test", X_test_all),
                    ("test sample of 1,000", X_test_all.sample(1000, random_state=RANDOM_STATE))]:
    t = psi_table(batch, reference)
    print(f"{name:22s} largest PSI {t['psi'].max():.4f} ({t['input'].iloc[0]}); {t['status'].value_counts().to_dict()}")

### 7.2 Synthetic batches: one like training, one shifted
The shifted batch imitates younger, thinner-file applicants: lower and more often missing external scores, shorter employment and fewer pensioners, more recent ID and address changes, 20% larger loans, more renters.

In [ ]:
batches = {name: pd.read_csv(path) for name, path in SAMPLE_PATHS.items()}
psi = pd.DataFrame({name: psi_table(b, reference).set_index("input")["psi"] for name, b in batches.items()})
psi["shifted status"] = psi["shifted"].map(lambda v: "alert" if v > 0.25 else ("watch" if v >= 0.10 else "stable"))
psi.sort_values("shifted", ascending=False).round(3)

In [ ]:
from src.validation import validate_batch

rows = {}
for name, b in batches.items():
    valid, errors = validate_batch(b, reference)
    scored = predict_batch(valid, artifacts)
    rows[name] = {"rows with input errors": len(errors), "mean PD": scored["pd"].mean(),
                  **scored["decision"].value_counts(normalize=True).reindex(["approve", "refer", "decline"]).to_dict()}
rows["test split"] = {"rows with input errors": 0, "mean PD": metadata["test_metrics"]["mean_pd"],
                      **metadata["test_metrics"]["decision_shares"]}
pd.DataFrame(rows).T.round(3)